# Public Data Indicators for Ex-Ante Financial Control: Procedure BG-RRP-1.015 "School STEM Environment"

**Element 2 (E2)** of the Financial Controller qualification thesis on *"Ex-Ante Control for Legality and Fraud and Irregularity Indicators in the Award and Expenditure of Funds under Procedure BG-RRP-1.015 'School STEM Environment'"*.

**Methodological safeguard:** A statistical indicator in public procurement records is **not** a proven legal violation. Indicators identify measurable patterns before commitment that require documentary verification against internal financial control files (Element 3) and preventive checklist controls (Element 4). In accordance with public registry rules, legal entity and school names and EIK/Bulstat identifiers are preserved throughout tables and analyses, while sole-trader (`ЕТ`) and natural-person names are redacted.

**Author:** Valeriy Beluhov.

## 1. Purpose, Scope, Inputs, and Relational Database Preparation

This study measures ex-ante control indicators verifiable prior to commitment in public data for procedure `BG-RRP-1.015` (National Recovery and Resilience Plan) under Art. 13(3) items 1 and 3 of the Financial Management and Control in the Public Sector Act (FMCPSA), Ministry of Finance Guidelines `ZMF-184/06.03.2020`, and Arts. 17, 20, and 21 of the Public Procurement Act (PPA).

**Working Hypothesis (H2):** Pre-specified decision rule — confirmed if at least two of the three contract-level indicators (`predated`, `split_30d`, `regime_mismatch`) exhibit a lower 95% Wilson confidence bound exceeding **5%** (`0.05`) across participating schools with contracts (`N = 2,222`).

**Primary Inputs (read-only):**
- `openData.xml` and `openData.xsd` (`EXTRACTION_DATE = "2026-09-10"`): ISUN 2020 open data export hashed in `data/MANIFEST.csv`;
- Upstream Element 1 (`adfi-violations-classification@c9bd90d`): `adfi_reports_meta.csv`, `adfi_classification.csv`, `adfi_classification_verbatim.csv`, `classifier_classes.csv`.


In [1]:
from __future__ import annotations

import csv
import math
import re
from difflib import SequenceMatcher
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import networkx as nx
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import sparse, stats
from statsmodels.discrete.discrete_model import Poisson

EXTRACTION_DATE = "2026-09-10"
PROCEDURE_PREFIX = "BG-RRP-1.015"
EUR_BGN = 1.95583
VAT_FACTOR = 1.20
RATIO_TOLERANCE = 0.03
REGIME_1_CUTOFF = "2023-12-31"
EURO_FROM = "2026-01-01"
RNG_STATE = 2026
WILSON_Z = 1.95996

DATA_DIR = Path("data")
DERIVED_DIR = DATA_DIR / "derived"
FIG_DIR = Path("fig")
DERIVED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.sans-serif": ["DejaVu Sans"],
    "axes.edgecolor": "#444444",
    "axes.linewidth": 0.8,
})

conclusions_log = []


def record_conclusion(part, question, measure, value, interval_low="", interval_high="", interval_method="none", decision="none"):
    """Append one verified quantitative metric row for conclusions_summary.csv."""
    conclusions_log.append({
        "part": str(part),
        "question": str(question),
        "measure": str(measure),
        "value": str(value),
        "interval_low": "" if interval_low == "" else str(interval_low),
        "interval_high": "" if interval_high == "" else str(interval_high),
        "interval_method": str(interval_method),
        "decision": str(decision),
    })


def flush_conclusions():
    """Persist conclusions_summary.csv using Element 1 semicolon-delimited format."""
    df_out = pd.DataFrame(
        conclusions_log,
        columns=["part", "question", "measure", "value", "interval_low", "interval_high", "interval_method", "decision"],
    )
    df_out.to_csv(DERIVED_DIR / "conclusions_summary.csv", sep=";", index=False, encoding="utf-8", lineterminator="\n")
    return df_out


def wilson_ci(k: int, n: int, z: float = WILSON_Z):
    """Return (proportion, lower_95, upper_95) using the Wilson score interval."""
    if n <= 0:
        return 0.0, 0.0, 0.0
    p = k / n
    denom = 1.0 + z * z / n
    center = (p + z * z / (2.0 * n)) / denom
    half = z * math.sqrt(p * (1.0 - p) / n + z * z / (4.0 * n * n)) / denom
    return p, max(0.0, center - half), min(1.0, center + half)


def bh_fdr(p_values):
    """Compute Benjamini-Hochberg adjusted q-values."""
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p, kind="stable")
    scaled = p[order] * len(p) / np.arange(1, len(p) + 1)
    q = np.empty_like(p)
    q[order] = np.minimum(np.minimum.accumulate(scaled[::-1])[::-1], 1.0)
    return q


In [2]:
isun_projects = pd.read_csv(DATA_DIR / "isun_projects.csv", dtype=str, keep_default_na=False)
for col in ("total_value", "actually_paid_amount"):
    isun_projects[col] = pd.to_numeric(isun_projects[col])

isun_contracts = pd.read_csv(DATA_DIR / "isun_contracts.csv", dtype=str, keep_default_na=False)
for col in ("amount_raw", "subcontractors_count"):
    isun_contracts[col] = pd.to_numeric(isun_contracts[col])
isun_contracts["subcontractors_count"] = isun_contracts["subcontractors_count"].astype(int)

isun_entities = pd.read_csv(DATA_DIR / "isun_entities.csv", dtype=str, keep_default_na=False)
schools_register = pd.read_csv(DATA_DIR / "schools_register.csv", dtype=str, keep_default_na=False)
for col in schools_register.columns[9:]:
    schools_register[col] = pd.to_numeric(schools_register[col])

adfi_meta = pd.read_csv(DATA_DIR / "adfi_reports_meta.csv", sep=";", dtype=str, keep_default_na=False)
adfi_cls = pd.read_csv(DATA_DIR / "adfi_classification.csv", sep=",", dtype=str, keep_default_na=False)
adfi_verb = pd.read_csv(DATA_DIR / "adfi_classification_verbatim.csv", sep=";", dtype=str, keep_default_na=False)
classifier_classes = pd.read_csv(DATA_DIR / "classifier_classes.csv", sep=";", dtype=str, keep_default_na=False)

contracts = pd.read_csv(DERIVED_DIR / "contracts.csv", dtype=str, keep_default_na=False)
for col in ("contract_row", "delta_t_days", "amount_raw", "amount_eur_incl_vat", "amount_eur_excl_vat", "amount_bgn_excl_vat", "paid_alloc_eur_incl_vat", "subcontractors_count"):
    contracts[col] = pd.to_numeric(contracts[col])

schools = pd.read_csv(DERIVED_DIR / "schools.csv", dtype=str, keep_default_na=False)
for col in ("projects_count", "contracts_count", "register_budget_bgn_incl_vat", "contracted_eur_incl_vat", "contracted_eur_excl_vat", "paid_eur_incl_vat"):
    schools[col] = pd.to_numeric(schools[col])
for col in ("flag_predated", "flag_split_30d", "flag_regime_mismatch", "flag_level1", "flag_bunching_50k"):
    schools[col] = schools[col] == "True"

project_classes = pd.read_csv(DERIVED_DIR / "project_classes.csv", dtype=str, keep_default_na=False)
for col in ("total_value", "actually_paid_amount", "contract_sum_raw", "register_budget_bgn_incl_vat", "register_budget_eur_incl_vat", "paid_eur_incl_vat"):
    project_classes[col] = pd.to_numeric(project_classes[col])

normalized_totals = pd.read_csv(DERIVED_DIR / "normalized_totals.csv")
flags_tbl = pd.read_csv(DERIVED_DIR / "forensic_risk_flags.csv", dtype=str, keep_default_na=False)
for col in ("contract_row", "amount_raw", "amount_eur_excl_vat", "amount_bgn_excl_vat", "threshold_bgn_excl_vat"):
    flags_tbl[col] = pd.to_numeric(flags_tbl[col])
for col in ("flag_predated", "flag_split_30d", "flag_same_day_sub", "flag_regime_mismatch", "flag_level1", "flag_bunching_30k", "flag_bunching_50k", "flag_bunching_80k"):
    flags_tbl[col] = flags_tbl[col] == "True"

assert len(isun_projects) == 2228
assert len(isun_contracts) == 11128
assert len(contracts) == 11128
assert len(schools) == 2222
assert "vendor_role" not in contracts.columns
assert "name" in isun_entities.columns and "vendor_name" in contracts.columns and "school_name" in schools.columns
assert not any("zdoi" in c.lower() for c in schools.columns)

print(f"Loaded {len(isun_projects):,} projects, {len(isun_contracts):,} contracts, {len(isun_entities):,} entities, {len(schools_register):,} register institutions.")
print(f"Loaded {len(adfi_meta):,} ADFI reports ({len(adfi_cls):,} classification rows) and {len(schools):,} STEM schools with contracts.")


Loaded 2,228 projects, 11,128 contracts, 3,973 entities, 2,271 register institutions.
Loaded 560 ADFI reports (1,824 classification rows) and 2,222 STEM schools with contracts.
